In [1]:
import numpy as np
import pandas as pd

from sklearn.model_selection import train_test_split
from sklearn.metrics import r2_score

import joblib

from lightgbm import LGBMRegressor
import lightgbm as lgb

from demand_model import DemandModel
from optimizer import tune

c:\Users\prasa\Desktop\NITD CSE\GridLock\.venv\Lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


In [2]:
df = pd.read_csv("dataset/train.csv", index_col='Index')
df.head()

,geohash,day,timestamp,demand,RoadType,NumberofLanes,LargeVehicles,Landmarks,Temperature,Weather
Index,,,,,,,,,,
0,qp02z1,48,0:0,0.048804,NaN,1,Not Allowed,No,NaN,NaN
1,qp02zt,48,0:0,0.118507,Residential,3,Allowed,Yes,31.104565,Sunny
2,qp08bj,48,0:0,0.027132,Residential,1,Not Allowed,No,25.919267,Sunny
3,qp08gt,48,0:0,0.003272,Residential,1,Not Allowed,No,NaN,Rainy
4,qp02zq,48,0:0,0.010819,Residential,1,Not Allowed,No,10.803667,Rainy


In [3]:
tv_df, test_df = train_test_split(
    df,
    test_size=0.2,
    random_state=42,
    shuffle=True
)

In [4]:
print(len(tv_df), len(test_df))

61839 15460


### -----HYPERPARAMETER TUNING----- ###

In [5]:
params_base = {
    "objective": "regression",
    "metric": "rmse",
    "verbosity": -1,
    "n_estimators": 5000
}

In [6]:
lightgbm_search_space = {

    "learning_rate": {
        "type": "float",
        "low": 0.005,
        "high": 0.05,
        "log": True
    },

    "num_leaves": {
        "type": "int",
        "low": 16,
        "high": 256
    },

    "max_depth": {
        "type": "int",
        "low": 3,
        "high": 12
    },

    "min_data_in_leaf": {
        "type": "int",
        "low": 20,
        "high": 200
    },

    "feature_fraction": {
        "type": "float",
        "low": 0.5,
        "high": 1.0
    },

    "bagging_fraction": {
        "type": "float",
        "low": 0.5,
        "high": 1.0
    },

    "bagging_freq": {
        "type": "int",
        "low": 1,
        "high": 10
    },

    "lambda_l1": {
        "type": "float",
        "low": 0.0,
        "high": 10.0
    },

    "lambda_l2": {
        "type": "float",
        "low": 0.0,
        "high": 20.0
    }
}

In [8]:
study = tune(
    df=tv_df,
    estimator_class=LGBMRegressor,
    search_space=lightgbm_search_space,
    params_base=params_base,
    fit_config={
        "eval_metric": "rmse",
        "callbacks": [
            lgb.early_stopping(100, verbose=False),
        ]
    },
    n_trials=50
)

best_params = study.best_params

print(best_params)

[I 2026-06-07 14:06:22,475] A new study created in memory with name: no-name-1e29e72c-55de-4198-a3da-c1c29fd12ede
[I 2026-06-07 14:06:34,676] Trial 0 finished with value: 0.9067182527645443 and parameters: {'learning_rate': 0.009292677381376041, 'num_leaves': 131, 'max_depth': 5, 'min_data_in_leaf': 21, 'feature_fraction': 0.7739216312736298, 'bagging_fraction': 0.9478627900297558, 'bagging_freq': 9, 'lambda_l1': 5.791599713060424, 'lambda_l2': 0.7296783733112577}. Best is trial 0 with value: 0.9067182527645443.
[I 2026-06-07 14:06:48,404] Trial 1 finished with value: 0.9053035528788836 and parameters: {'learning_rate': 0.028298859379520376, 'num_leaves': 222, 'max_depth': 10, 'min_data_in_leaf': 192, 'feature_fraction': 0.5478575117220167, 'bagging_fraction': 0.5988350414236585, 'bagging_freq': 4, 'lambda_l1': 2.956437562431302, 'lambda_l2': 17.324767526453144}. Best is trial 0 with value: 0.9067182527645443.
[I 2026-06-07 14:06:58,613] Trial 2 finished with value: 0.9061515088232053 


Best Score: 0.9169559188782628

Best Parameters:
learning_rate: 0.008856221455006523
num_leaves: 211
max_depth: 10
min_data_in_leaf: 22
feature_fraction: 0.9604802847525437
bagging_fraction: 0.8755168105261125
bagging_freq: 1
lambda_l1: 0.8087035218903588
lambda_l2: 3.0083665010425586
{'learning_rate': 0.008856221455006523, 'num_leaves': 211, 'max_depth': 10, 'min_data_in_leaf': 22, 'feature_fraction': 0.9604802847525437, 'bagging_fraction': 0.8755168105261125, 'bagging_freq': 1, 'lambda_l1': 0.8087035218903588, 'lambda_l2': 3.0083665010425586}


In [9]:
from sklearn.model_selection import KFold

x = tv_df.copy()
y = tv_df["demand"]

kf = KFold(n_splits=4,
           shuffle=True,
           random_state=42)
oof = np.zeros(len(tv_df))

for fold, (tr, va) in enumerate(kf.split(x, y)):

    train_fold = tv_df.iloc[tr]
    val_fold = tv_df.iloc[va]

    model = DemandModel(
        estimator_class=LGBMRegressor,
        params={
            **params_base,
            **best_params
        },
        fit_config={
            "eval_metric": "rmse",
            "callbacks": [
                lgb.early_stopping(100),
                lgb.log_evaluation(50)
            ]
        },
        verbose=True
    )
    
    model.fit(train_fold, val_fold)

    preds = model.predict(val_fold)
    oof[va] = preds

    print(f"Fold {fold + 1} R2:", r2_score(val_fold["demand"], preds))

print("OOF R2:", r2_score(y, oof))

Training until validation scores don't improve for 100 rounds
[50]	valid_0's rmse: 0.103054
[100]	valid_0's rmse: 0.0757016
[150]	valid_0's rmse: 0.0602577
[200]	valid_0's rmse: 0.0519369
[250]	valid_0's rmse: 0.0475548
[300]	valid_0's rmse: 0.0453039
[350]	valid_0's rmse: 0.0440472
[400]	valid_0's rmse: 0.0432262
[450]	valid_0's rmse: 0.0427234
[500]	valid_0's rmse: 0.0424234
[550]	valid_0's rmse: 0.0421938
[600]	valid_0's rmse: 0.0420275
[650]	valid_0's rmse: 0.0419142
[700]	valid_0's rmse: 0.0418292
[750]	valid_0's rmse: 0.0417631
[800]	valid_0's rmse: 0.0416827
[850]	valid_0's rmse: 0.0416244
[900]	valid_0's rmse: 0.0415787
[950]	valid_0's rmse: 0.0415252
[1000]	valid_0's rmse: 0.0414808
[1050]	valid_0's rmse: 0.0414562
[1100]	valid_0's rmse: 0.0414303
[1150]	valid_0's rmse: 0.0414068
[1200]	valid_0's rmse: 0.0413752
[1250]	valid_0's rmse: 0.0413498
[1300]	valid_0's rmse: 0.0413322
[1350]	valid_0's rmse: 0.0413097
[1400]	valid_0's rmse: 0.041293
[1450]	valid_0's rmse: 0.041285
[150

### -----TESTING ROUTINE----- ###

In [10]:
test_y = test_df["demand"]
test_x = test_df.drop(columns=["demand"])

preds = model.predict(test_x)
print("Test R2:", r2_score(test_y, preds))

Test R2: 0.9192369215726754


In [10]:
joblib.dump(model.model, "model.pkl")
joblib.dump(model.pipeline, "pipeline.pkl")

['pipeline.pkl']

In [11]:
submission_test_df = pd.read_csv("dataset/test.csv")
predictions = model.predict(submission_test_df)

submission = pd.DataFrame({
    "Index": submission_test_df.index,
    "demand": predictions
})

submission.to_csv("dataset/submission.csv", index=False)